In [3]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [4]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
# random sampling

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

# Data Pre-Processing

In [11]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenize

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [16]:
# raw data => tokenize inputs for fine tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [17]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [19]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# input ids
# 1 => end of sequence
# attention mask
# labels - target =>summary token

In [20]:
len(train_dataset[0]["input_ids"])

512

In [21]:
type(train_dataset)
type(val_dataset)

list

 # Working with our model
 

In [22]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [23]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [27]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.633400,0.379591
2,0.396485,0.359951
3,0.372924,0.354922
4,0.362279,0.351234
5,0.354450,0.349991
6,0.351070,0.349361


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9117678731282552, metrics={'train_runtime': 1978.5259, 'train_samples_per_second': 12.13, 'train_steps_per_second': 1.516, 'total_flos': 3248203235328000.0, 'train_loss': 0.9117678731282552, 'epoch': 6.0})

In [28]:
# model load => fine tune => save this model

In [29]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [30]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [31]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    #tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    #token ids convert to summary => decoding, decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [32]:
test_dialogue = """ Reporter: Good morning, everyone. Today we have an AI expert with us to discuss how Artificial Intelligence is being used in today's world. Welcome!

Expert: Thank you for having me. I'm happy to be here.

Reporter: First, could you explain what Artificial Intelligence is?

Expert: Artificial Intelligence, or AI, is a technology that enables computers and machines to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving, and understanding language.

Reporter: AI seems to be everywhere these days. Where do we use it in our daily lives?

Expert: AI is used in many areas. For example, voice assistants like Siri and Google Assistant, recommendation systems on YouTube and Netflix, navigation apps like Google Maps, online shopping websites, and even spam filters in email all use AI.

Reporter: How is AI helping students?

Expert: AI helps students by providing personalized learning experiences, answering questions, generating study materials, translating languages, and offering virtual tutoring. It can make learning more accessible and efficient.

Reporter: What role does AI play in healthcare?

Expert: In healthcare, AI assists doctors in diagnosing diseases, analyzing medical images, predicting health risks, and managing patient records. This can improve accuracy and save time.

Reporter: How is AI benefiting businesses?

Expert: Businesses use AI to automate repetitive tasks, analyze customer behavior, improve customer service through chatbots, and make data-driven decisions. This increases productivity and efficiency.

Reporter: Are there any challenges associated with AI?

Expert: Yes. Some challenges include data privacy concerns, potential job displacement due to automation, bias in AI systems, and the need for responsible use of AI technologies.

Reporter: What do you think the future of AI looks like?

Expert: AI will continue to evolve and become more integrated into our lives. It has the potential to improve education, healthcare, transportation, and many other fields. However, it is important to develop and use AI responsibly.

Reporter: Finally, what message would you like to give to our audience?

Expert: AI is a powerful tool that can help solve many problems and improve our quality of life. Learning about AI and using it responsibly will help us make the most of its benefits.

Reporter: Thank you for sharing your insights with us today.

Expert: Thank you. It was a pleasure speaking with you.

Reporter: Thank you, everyone, for joining us. Have a great day! """


summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary)

Summary:  expert has an ai expert with him to discuss how artificial intelligence is being used in today's world. ai helps students by providing personalized learning experiences, answering questions, translating languages, and offering virtual tutoring.
